# 🕵️ Bank Account Fraud - Notebook 02: Fraud Deep-Dive & Pattern Analysis
Phân tích chuyên sâu các mô hình gian lận mở tài khoản ngân hàng.

### Nội dung phân tích:
1. Xu hướng tỷ lệ gian lận theo thời gian (Months 0 - 7).
2. Phân tích Dấu chân kỹ thuật số (Device OS, Session length, Free Email).
3. Phân tích Động thái cư trú & Tín dụng (Housing Status, Employment, Credit limit).
4. Xây dựng và đánh giá các bộ quy tắc phát hiện gian lận (Heuristic Fraud Rules).

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append("..")
from src.visualization.visualize import apply_theme, plot_fraud_by_category, plot_monthly_trend

df = pd.read_parquet("../data/processed/clean_base.parquet")
print(f"Đã nạp {len(df):,} hồ sơ mở tài khoản.")

## 1. Xu hướng gian lận qua các tháng (Monthly Trend)

In [ ]:
monthly_stats = df.groupby("month").agg(
    total_apps=("fraud_bool", "count"),
    fraud_cases=("fraud_bool", "sum"),
    fraud_rate=("fraud_bool", lambda x: x.mean() * 100)
).reset_index()
monthly_stats

## 2. Rủi ro theo Hệ điều hành & Tình trạng cư trú

In [ ]:
plot_fraud_by_category(df, "device_os", "Tỷ lệ gian lận theo Hệ điều hành")
plot_fraud_by_category(df, "housing_status", "Tỷ lệ gian lận theo Tình trạng nhà ở")

## 3. Đánh giá Bộ Quy Tắc Phát Hiện Gian Lận (Heuristic Rules)
Đo lường tỷ lệ bắt gian lận (Precision / Hit Rate) so với tỷ lệ cơ sở (1.10%):

In [ ]:
# Rule 1: Windows + Housing BA + Không có địa chỉ cũ
rule1 = (df["device_os"] == "windows") & (df["housing_status"] == "BA") & (df["has_prev_address"] == 0)

# Rule 2: Tên email không khớp (<0.3) + Email miễn phí + Yêu cầu từ nước ngoài
rule2 = (df["name_email_similarity"] < 0.3) & (df["email_is_free"] == 1) & (df["foreign_request"] == 1)

# Rule 3: Không có tài khoản cũ + Không có số điện thoại hợp lệ
rule3 = (df["has_prior_bank_account"] == 0) & (df["no_valid_phone"] == 1)

results = []
for name, mask in [("Rule 1: Windows + Housing BA + No Prev Addr", rule1),
                    ("Rule 2: Low Similarity + Free Email + Foreign", rule2),
                    ("Rule 3: No Prior Bank + No Valid Phone", rule3)]:
    triggered = mask.sum()
    caught = df.loc[mask, "fraud_bool"].sum()
    precision = (caught / triggered * 100) if triggered > 0 else 0
    results.append({"Rule": name, "Triggered": int(triggered), "Frauds_Caught": int(caught), "Precision (%)": round(precision, 2)})

pd.DataFrame(results)